In [156]:
from dotenv import load_dotenv
load_dotenv()

True

In [158]:
from langchain_groq import ChatGroq 
from langgraph.graph import START,END,StateGraph
from langgraph.types import interrupt,Command
from typing import TypedDict,NotRequired,Annotated,Literal
from pydantic import Field,BaseModel
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver


In [159]:
from pydantic import BaseModel, ConfigDict, Field

class graphstate(BaseModel):
    model_config = ConfigDict(validate_assignment=True)
    topic: str
    email: str = ""
    final_email: str = ""
    human_fb: str = ""
    revisions: list[str] = Field(default_factory=list)

In [160]:
llm=ChatGroq(model="openai/gpt-oss-20b")  

In [161]:
def draft_mail(state:graphstate):
    if state.human_fb:
        res=llm.invoke(f"""You are an assistant which has generated email for a topic given by user , modify the email according to user's feedback
                            feedback by user:{state.human_fb}
                            topic:{state.topic}
                            Ai draft:{state.email}    """)
    else:
        res=llm.invoke(f"Generate an email according to the requirements given by the user ,query:{state.topic}")
    return {"email":res.content}

In [162]:
def human_feedback(state:graphstate):
    feedback=interrupt(
        {
        "Draft_mail":state.email,
        "Question":"Do u want to rewrite the email?"
        }
    )
    fb=(feedback or "").strip().lower()
    if fb in ["approved","yes","works"]:
        state.human_fb=""
    else :
        state.human_fb=feedback

    return state


In [163]:
def send_final(state:graphstate):
    if state.human_fb:
        state.final_email=state.human_fb
    else :
        print("Email sent successfully\n")
        state.final_email="Email sent"
    return state

In [164]:
def router(state:graphstate)->Literal["draft","final"]:
    if state.human_fb:
        return "draft"
    return "final"

In [165]:
graph=StateGraph(graphstate)
graph.add_node("draft",draft_mail)

In [166]:

graph.add_node("final",send_final)
graph.add_node("feedback",human_feedback)
graph.add_edge(START,"draft")
graph.add_edge("draft","feedback")
graph.add_conditional_edges("feedback",router)
graph.add_edge("final",END)

In [167]:
final_graph=graph.compile(checkpointer=InMemorySaver())
config={"configurable":{"thread_id":"1"}}

In [168]:
final_graph.invoke({"topic":"write an email to hr managermfor leave due to marriage in family from 1 dec to 12 dec 2026"},config)

{'topic': 'write an email to hr managermfor leave due to marriage in family from 1 dec to 12 dec 2026',
 'email': '**Subject:** Leave Request (1\u202fDec\u202f–\u202f12\u202fDec\u202f2026) for Family Wedding\n\nDear [HR Manager’s Name],\n\nI hope this message finds you well. I am writing to formally request a leave of absence from **1\u202fDecember\u202f2026 to 12\u202fDecember\u202f2026** due to a family wedding that I will be attending.  \n\nDuring this period, I will ensure that all of my current responsibilities are covered. I have already briefed [Colleague’s Name] on the status of the [Project/Task] and will provide a detailed hand‑over document by the end of this week. I will also be available via email and phone for any urgent matters that may arise.\n\nI kindly ask you to approve this leave request at your earliest convenience. Please let me know if you need any further information or if there are any forms I should complete.\n\nThank you for your understanding and support.\n\

In [169]:
res=final_graph.invoke(Command(resume="No , modify the reason to engagement of friend"),config=config)

In [170]:
print(res["email"])

**Subject:** Leave Request (1 Dec – 12 Dec 2026) for Friend’s Engagement

Dear [HR Manager’s Name],

I hope you’re doing well. I am writing to formally request a leave of absence from **1 December 2026 to 12 December 2026** to attend the engagement ceremony of a close friend.

During this period, I will ensure that all of my current responsibilities are covered. I have already briefed [Colleague’s Name] on the status of the [Project/Task] and will provide a detailed hand‑over document by the end of this week. I will also be available via email and phone for any urgent matters that may arise.

I kindly ask you to approve this leave request at your earliest convenience. Please let me know if you need any further information or if there are any forms I should complete.

Thank you for your understanding and support.

Warm regards,

[Your Full Name]  
[Your Designation] – [Department]  
[Employee ID]  
[Phone Number]  
[Email Address]
